In [1]:
import os
import logging
import pandas as pd

root_logger = logging.getLogger()
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)

file_handler = logging.FileHandler("pipeline.log", mode="w")
stream_handler = logging.StreamHandler()
log_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler.setFormatter(log_formatter)
stream_handler.setFormatter(log_formatter)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("Starting the traffic data pipeline.")

def load_csv_data(file_name):
    logger.info("Attempting to load the CSV file.")
    if not os.path.exists(file_name):
        logger.error("Error: The CSV file was not found in this folder.")
        return None
    try:
        data = pd.read_csv(file_name)
        rows, cols = data.shape
        logger.info(f"Success: CSV file loaded into memory. Loaded {rows} rows and {cols} columns.")
        return data
    except pd.errors.ParserError:
        logger.error("Error: The CSV file formatting is corrupted.")
        return None
    except Exception as e:
        logger.error(f"An unexpected error happened: {e}")
        return None

def validate_schema(df):
    if df is None:
        return False
    logger.info("Validating the data schema columns.")
    expected_columns = [
        'holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 
        'weather_main', 'weather_description', 'date_time', 'traffic_volume'
    ]
    missing_columns = []
    for col in expected_columns:
        if col not in df.columns:
            missing_columns.append(col)
    if len(missing_columns) > 0:
        logger.error(f"Schema Validation Failed! Missing columns: {missing_columns}")
        return False
    logger.info("Schema Validation Successful: All expected columns are present.")
    return True

def clean_data(df):
    if df is None:
        return None
    logger.info("Starting data cleaning process.")
    initial_rows = len(df)
    df = df.drop_duplicates()
    duplicate_count = initial_rows - len(df)
    logger.info(f"Removed {duplicate_count} duplicate rows.")
    df['weather_main'] = df['weather_main'].str.strip().str.title()
    df['weather_description'] = df['weather_description'].str.strip().str.lower()
    df['holiday'] = df['holiday'].str.strip()
    logger.info("Standardised categorical text values.")
    try:
        df['date_time'] = pd.to_datetime(df['date_time'])
        logger.info("Parsed and validated date/time fields.")
    except Exception as e:
        logger.error(f"Date/time parsing failed: {e}")
    df['month'] = df['date_time'].dt.month
    unique_months = sorted(df['month'].dropna().unique())
    for m in unique_months:
        month_mask = df['month'] == m
        valid_temp_mask = month_mask & (df['temp'] > 0)
        valid_month_temps = df.loc[valid_temp_mask, 'temp']
        if not valid_month_temps.empty:
            monthly_median_temp = valid_month_temps.median()
            zero_temp_mask = month_mask & (df['temp'] == 0)
            zero_count = zero_temp_mask.sum()
            if zero_count > 0:
                df.loc[zero_temp_mask, 'temp'] = monthly_median_temp
                logger.info(f"Imputed {zero_count} temperature outliers for month {m} using monthly median ({monthly_median_temp}K).")
        valid_rain_mask = month_mask & (df['rain_1h'] < 9000)
        valid_month_rain = df.loc[valid_rain_mask, 'rain_1h']
        if not valid_month_rain.empty:
            monthly_median_rain = valid_month_rain.median()
            extreme_rain_mask = month_mask & (df['rain_1h'] >= 9000)
            extreme_count = extreme_rain_mask.sum()
            if extreme_count > 0:
                df.loc[extreme_rain_mask, 'rain_1h'] = monthly_median_rain
                logger.info(f"Imputed {extreme_count} rainfall outliers for month {m} using monthly median ({monthly_median_rain}mm).")
    df = df.drop(columns=['month'])
    return df

def clean_and_transform(df):
    if df is None:
        logger.warning("No data found to process. Stopping here.")
        return None
    logger.info("Starting data checks.")
    missing_data = df.isnull().sum()
    logger.info("Checking for missing values in each column:")
    for column, count in missing_data.items():
        if count > 0:
            logger.warning(f"Column '{column}' has {count} missing rows.")
    logger.info("Checking column data types.")
    logger.info("Extracting hour from date_time column.")
    try:
        df['hour'] = df['date_time'].dt.hour
    except Exception as e:
        logger.error(f"Could not extract hour dimension: {e}")
    logging.info("Converting temperature from Kelvin to Celsius.")
    df['temp_celsius'] = df['temp'] - 273.15
    logger.info("Creating traffic category column.")
    def check_volume(volume):
        if volume > 5500:
            return 'High'
        elif volume >= 2500:
            return 'Medium'
        else:
            return 'Low'
    df['traffic_category'] = df['traffic_volume'].apply(check_volume)
    logger.info("All data pipeline transformations completed successfully.")
    return df

if __name__ == "__main__":
    my_file = "Metro_Interstate_Traffic_Volume.csv"
    raw_df = load_csv_data(my_file)
    if validate_schema(raw_df):
        cleaned_df = clean_data(raw_df)
        final_df = clean_and_transform(cleaned_df)
    else:
        logger.error("Pipeline stopped because schema validation failed.")


2026-09-24 16:48:27,730 - INFO - Starting the traffic data pipeline.
2026-09-24 16:48:27,731 - INFO - Attempting to load the CSV file.
2026-09-24 16:48:27,768 - INFO - Success: CSV file loaded into memory. Loaded 48204 rows and 9 columns.
2026-09-24 16:48:27,768 - INFO - Validating the data schema columns.
2026-09-24 16:48:27,769 - INFO - Schema Validation Successful: All expected columns are present.
2026-09-24 16:48:27,769 - INFO - Starting data cleaning process.
2026-09-24 16:48:27,786 - INFO - Removed 17 duplicate rows.
C:\Users\User\AppData\Local\Temp\ipykernel_25740\610189956.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['weather_main'] = df['weather_main'].str.strip().str.title()
C:\Users\User\AppData\Local\Temp\ipykernel_25740\6